In [0]:
%run ../config/set_up_env_paths

In [0]:
from config.core import config
from pyspark.sql.functions import col , when, count


In [0]:
table_a = spark.table(f"{config.catalog.catalog_name}.{config.catalog.bronze_schema}.{config.nbp.table_a}")



In [0]:
table_a.describe().display()

summary,date,code,price
count,726,726,726
mean,null,null,1.249135961997244
stddev,null,null,1.5862365939536225
min,2025-10-20,AUD,2.1764E-4
max,2025-11-19,ZAR,5.0094


In [0]:

null_counts = [ count( when(col(c).isNull(), 1)).alias(c)  for c in table_a.columns  ]


In [0]:
display(table_a.select(*null_counts))

date,code,price
0,0,0


In [0]:
# No duplicates 
table_a.count(), table_a.dropDuplicates().count()

(726, 726)

In [0]:
table_a.filter(col("price")<0).display()

date,code,price


# Table B

In [0]:
table_b = spark.table(f"{config.catalog.catalog_name}.{config.catalog.bronze_schema}.{config.nbp.table_b}")

table_b.printSchema()

root
 |-- date: string (nullable = true)
 |-- code: string (nullable = true)
 |-- price: double (nullable = true)



In [0]:
table_b.describe().display()

summary,date,code,price
count,580,580,580
mean,null,null,0.7881339550517241
stddev,null,null,1.8595468527808552
min,2025-10-22,AED,4.1E-5
max,2025-11-19,ZWG,12.0723


In [0]:
null_counts = [ count( when(col(c).isNull(), 1)).alias(c)  for c in table_b.columns  ]
display(table_b.select(*null_counts))

date,code,price
0,0,0


In [0]:
table_b.count(), table_b.dropDuplicates().count()

(580, 580)

In [0]:
table_b.filter(col("price")<0).display()

date,code,price


## Findings:
1. Date is string, we should convert it into DateTime object  so later to extract valuable data
2. Name of `code` column should be changed as  `currency_code` 
3. Name of `price` column should be changed as `price_in_PLN` 
4.  `price_PLN` should be rounded to 2 decimal digits 
5. Need to take under consideration that what if we get null data from API or data will be invalid like `price_PLN` < 0